# 1: Import File into Dataframe

In the code below we import the dataframe into pandas, and remove some weird NaN value columns. We wil also filter the dataset to only include sets with a retail price higher than £20. This was done because data analysis showed that the profit margin on sets with low values is very erratic and hard to predict. This makes logical sense, because a price rise of 100% from £5 to £10, is a lot easier for a consumer to stomach than a rise from £20 to £40.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import pandas

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

df = pandas.read_csv("/kaggle/input/lego-model-information-and-return-on-investment/lego-returns-kaggle.csv")
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df[df['retail_price'] > 20]

# 2: Mutate Dataframe to Add Returns Column

In [ ]:
df['return'] = df['pop_price']/df['retail_price']

# 3: Split Dataframe into Two Datasets

Split into two datasets. One will be used to train, the other as a held-out dataset to test at the end. We will split at a ratio of 80/20 train-test.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2)

print(len(train_df), len(test_df))

# 4: Train Regression Model

Here we will train the regression model. We will evaluate using five-fold cross evaluation. We will then find the overall R-squared score for the model, and plot the predicted vs real price changes.

In [ ]:
from sklearn import linear_model
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import make_scorer
import matplotlib.pyplot as plt

X = train_df.drop(['return', 'pop_price', 'set_id', 'set_name', 'retire_month'], axis=1)
y = train_df['return']

model = linear_model.LinearRegression()

# Display the cross-validation R2 score
cross_val_scores = cross_val_score(model, X, y, cv=5)
print("Cross-Validation R2 Score:", cross_val_scores.mean())

# Display the cross-validation MAE score
mae = make_scorer(mean_absolute_error, greater_is_better=False)
cross_val_scores = cross_val_score(model, X, y, cv=5, scoring=mae)
print("Cross-Validation MAE Score:", -cross_val_scores.mean())

# Display plot of predicted y and true y values
predicted_y = cross_val_predict(model, X, y, cv=5)
plt.scatter(y, predicted_y, alpha=0.5)

# 5: Preliminary Analysis

As can be seen above, the preliminary analysis does not bode well. The R-squared score is only 0.25. This is not great, however it does show some correlation.

Our Mean Absolute Error score is 0.22, meaning on average, our predictions are 22% off the real value. This is not great, but also not terrible for a simple model.

# 6: Final Analysis

Final analysis of the model with the held out dataset. We will essentially do the same thing as in the previous step, but with the other dataset. 

Note: It may appear that no iterative development has occurred in this sheet, negating the need for a held-out dataset. However this project was worked on for a week or so, and behind the scenes the first model was developed over many iterations, with lots of different columns of scraped data. In this sheet I have only included the columns from the final model.

In [ ]:
from sklearn.metrics import r2_score

X = train_df.drop(['return', 'pop_price', 'set_id', 'set_name', 'retire_month'], axis=1)
y = train_df['return']

model = linear_model.LinearRegression()
regr = linear_model.LinearRegression()
regr.fit(X, y)

y_pred = regr.predict(test_df.drop(['return', 'pop_price', 'set_id', 'set_name', 'retire_month'], axis=1))
y_true = test_df['return'] 

print("Final R2 Score:", r2_score(y_true, y_pred))
print("Final MAE Score:", mean_absolute_error(y_true, y_pred))
plt.scatter(y_true, y_pred, alpha=0.5)

# 7: Conclusions

As a result of the final analysis, we can see that the performance of the model is not amazing. With final MAE and R2 hovering around 0.2, this model could not be used to reliably predict the price increase of a lego set 1 year from retirement. 

We could conclude that other variables must be needed, however many other variables were tested in development, scraped from various sources. These include:

* Number of Reddit posts relating to each set
* Number of EuroBricks posts relating to each set
* Average return of the other sets in the theme
* Rarity of pieces in each set (An average of: For each piece in the set, how many sets share said piece)

Despite seeming promising, none of these features improved the accuracy significantly. So we can conclude that Lego sets must be fairly erratic in pricing. It is likely that price increase is determined by a myriad of factors, many of which are not quantifiable. e.g. Collectibility.